# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials.parquet')

In [5]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.004922,0.001773,0.993304,0.004134,0.002593,0.993273
1,0.997228,0.000463,0.002309,0.995591,0.000405,0.004004
2,0.002369,0.000604,0.997026,0.001958,0.000503,0.997538
3,0.004656,0.003362,0.991982,0.002466,0.002391,0.995142
4,0.995353,0.000016,0.004632,0.994178,0.000003,0.005819


In [6]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012452,0.003199,0.984349,0.011048,0.004491,0.984461
1,0.472116,0.002331,0.525553,0.499186,0.001896,0.498918
2,0.998690,0.001012,0.000298,0.997826,0.001594,0.000579
3,0.980854,0.000018,0.019128,0.987434,0.000009,0.012557
4,0.006765,0.001847,0.991388,0.006876,0.001972,0.991152


In [7]:
y_train.head()

,health_condition
id,
0,2
1,0
2,2
3,2
4,0


# Machine Learning

In [8]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [9]:
def objective(trial, X, y):

    lgbm_w0 = trial.suggest_float('weight_lgbm_class_0', 0.0, 1.0)
    lgbm_w1 = trial.suggest_float('weight_lgbm_class_1', 0.0, 1.0)
    lgbm_w2 = trial.suggest_float('weight_lgbm_class_2', 0.0, 1.0)
    
    xgb_w0 = trial.suggest_float('weight_xgb_class_0', 0.0, 1.0)
    xgb_w1 = trial.suggest_float('weight_xgb_class_1', 0.0, 1.0)
    xgb_w2 = trial.suggest_float('weight_xgb_class_2', 0.0, 1.0)
    
    # cat_w0 = trial.suggest_float('weight_cat_class_0', 0.0, 1.0)
    # cat_w1 = trial.suggest_float('weight_cat_class_1', 0.0, 1.0)
    # cat_w2 = trial.suggest_float('weight_cat_class_2', 0.0, 1.0)
    
    proba_lgbm = X[['lgbm_0', 'lgbm_1', 'lgbm_2']].to_numpy()
    proba_xgb  = X[['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    # proba_cat  = X[['cat_0', 'cat_1', 'cat_2']].to_numpy()

    w_lgbm = np.array([lgbm_w0, lgbm_w1, lgbm_w2])
    w_xgb  = np.array([xgb_w0, xgb_w1, xgb_w2])
    # w_cat  = np.array([cat_w0, cat_w1, cat_w2])
    
    blend_proba = (proba_lgbm * w_lgbm) + (proba_xgb * w_xgb) # + (proba_cat * w_cat)
    
    pred = np.argmax(blend_proba, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=2000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-05 15:31:31,403] A new study created in memory with name: no-name-034458e4-c8bd-4b33-8ded-8758fa325dac
                                                                                                                                                                                                                   

[I 2026-07-05 15:31:31,600] Trial 0 finished with value: 0.8759317873440423 and parameters: {'weight_lgbm_class_0': 0.37110426259962925, 'weight_lgbm_class_1': 0.7667893969991753, 'weight_lgbm_class_2': 0.3870816026439209, 'weight_xgb_class_0': 0.8987224659612689, 'weight_xgb_class_1': 0.5844597872283822, 'weight_xgb_class_2': 0.6620972639911515}. Best is trial 0 with value: 0.8759317873440423.
[I 2026-07-05 15:31:31,620] Trial 5 finished with value: 0.8740848614910811 and parameters: {'weight_lgbm_class_0': 0.5609230725323702, 'weight_lgbm_class_1': 0.31730592558460113, 'weight_lgbm_class_2': 0.6781136203704915, 'weight_xgb_class_0': 0.09698154555303051, 'weight_xgb_class_1': 0.07084341947817996, 'weight_xgb_class_2': 0.11830012684951408}. Best is trial 0 with value: 0.8759317873440423.
[I 2026-07-05 15:31:31,626] Trial 4 finished with value: 0.8467341294934693 and parameters: {'weight_lgbm_class_0': 0.0373100738319081, 'weight_lgbm_class_1': 0.06527270877525782, 'weight_lgbm_class_2'

Best trial: 15. Best value: 0.942462:   1%|█▎                                                                                                                                    | 19/2000 [00:00<00:54, 36.24it/s]

[I 2026-07-05 15:31:31,647] Trial 1 finished with value: 0.9038460381975165 and parameters: {'weight_lgbm_class_0': 0.19876765697596588, 'weight_lgbm_class_1': 0.8366848626402802, 'weight_lgbm_class_2': 0.008206597684362604, 'weight_xgb_class_0': 0.3886423133116683, 'weight_xgb_class_1': 0.9730547996152823, 'weight_xgb_class_2': 0.5460402937409152}. Best is trial 10 with value: 0.9395374134151561.
[I 2026-07-05 15:31:31,648] Trial 6 finished with value: 0.9224504378173392 and parameters: {'weight_lgbm_class_0': 0.21118493876401734, 'weight_lgbm_class_1': 0.8528518532638305, 'weight_lgbm_class_2': 0.502835392382978, 'weight_xgb_class_0': 0.4865591315081631, 'weight_xgb_class_1': 0.7022562854103667, 'weight_xgb_class_2': 0.8388633220836711}. Best is trial 10 with value: 0.9395374134151561.
[I 2026-07-05 15:31:31,769] Trial 12 finished with value: 0.8613484270972153 and parameters: {'weight_lgbm_class_0': 0.6810664911133526, 'weight_lgbm_class_1': 0.32667323334537346, 'weight_lgbm_class_2

Best trial: 32. Best value: 0.943075:   2%|██                                                                                                                                    | 30/2000 [00:00<00:46, 42.56it/s]

...

[I 2026-07-05 15:32:37,607] Trial 1982 finished with value: 0.9494680210622578 and parameters: {'weight_lgbm_class_0': 0.12174072904206529, 'weight_lgbm_class_1': 0.9835980213252691, 'weight_lgbm_class_2': 0.4018989203118749, 'weight_xgb_class_0': 8.49734006977763e-05, 'weight_xgb_class_1': 0.11665158551530458, 'weight_xgb_class_2': 0.6155371635944799}. Best is trial 1561 with value: 0.9498642565074636.
[I 2026-07-05 15:32:37,625] Trial 1983 finished with value: 0.9450772998589471 and parameters: {'weight_lgbm_class_0': 0.12995004939376986, 'weight_lgbm_class_1': 0.9831061731800851, 'weight_lgbm_class_2': 0.4027729838303132, 'weight_xgb_class_0': 0.046652860768569324, 'weight_xgb_class_1': 0.7587514267427448, 'weight_xgb_class_2': 0.2185692243411408}. Best is trial 1561 with value: 0.9498642565074636.
[I 2026-07-05 15:32:37,717] Trial 1984 finished with value: 0.948715968711853 and parameters: {'weight_lgbm_class_0': 0.12085540228840388, 'weight_lgbm_class_1': 0.982899471266262, 'weigh

Best trial: 1561. Best value: 0.949864: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 1992/2000 [01:06<00:00, 18.55it/s]

[I 2026-07-05 15:32:37,947] Trial 1987 finished with value: 0.9482735125691528 and parameters: {'weight_lgbm_class_0': 0.12440262196916119, 'weight_lgbm_class_1': 0.9842854000296758, 'weight_lgbm_class_2': 0.40470010615295204, 'weight_xgb_class_0': 0.05195320343877389, 'weight_xgb_class_1': 0.7256989717584661, 'weight_xgb_class_2': 0.5289170864654839}. Best is trial 1561 with value: 0.9498642565074636.
[I 2026-07-05 15:32:38,027] Trial 1988 finished with value: 0.9484663607569352 and parameters: {'weight_lgbm_class_0': 0.12429217916818959, 'weight_lgbm_class_1': 0.9998140063225593, 'weight_lgbm_class_2': 0.3717930183532809, 'weight_xgb_class_0': 0.04531859757486013, 'weight_xgb_class_1': 0.724505861812348, 'weight_xgb_class_2': 0.5793638252343297}. Best is trial 1561 with value: 0.9498642565074636.
[I 2026-07-05 15:32:38,055] Trial 1990 finished with value: 0.9498286275169319 and parameters: {'weight_lgbm_class_0': 0.06365744559875179, 'weight_lgbm_class_1': 0.9995857322283699, 'weight

Best trial: 1999. Best value: 0.949868: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [01:06<00:00, 29.91it/s]


[I 2026-07-05 15:32:38,157] Trial 1992 finished with value: 0.949617227394261 and parameters: {'weight_lgbm_class_0': 0.06427147119725747, 'weight_lgbm_class_1': 0.9998942768526572, 'weight_lgbm_class_2': 0.37345987900618954, 'weight_xgb_class_0': 0.045678950671587434, 'weight_xgb_class_1': 0.720311071216761, 'weight_xgb_class_2': 0.5254212928468864}. Best is trial 1561 with value: 0.9498642565074636.
[I 2026-07-05 15:32:38,164] Trial 1994 finished with value: 0.9498297873725008 and parameters: {'weight_lgbm_class_0': 0.06698794726248766, 'weight_lgbm_class_1': 0.9504744399978441, 'weight_lgbm_class_2': 0.48251586013750225, 'weight_xgb_class_0': 0.0463661646126676, 'weight_xgb_class_1': 0.7188006593590205, 'weight_xgb_class_2': 0.5266348837133208}. Best is trial 1561 with value: 0.9498642565074636.
[I 2026-07-05 15:32:38,203] Trial 1995 finished with value: 0.947580354056648 and parameters: {'weight_lgbm_class_0': 0.15442576812301986, 'weight_lgbm_class_1': 0.95511182336681, 'weight_lg

In [10]:
best_params = study.best_params

w_lgbm = np.array([best_params['weight_lgbm_class_0'], best_params['weight_lgbm_class_1'], best_params['weight_lgbm_class_2']])
w_xgb  = np.array([best_params['weight_xgb_class_0'],  best_params['weight_xgb_class_1'],  best_params['weight_xgb_class_2']])
# w_cat  = np.array([best_params['weight_cat_class_0'],  best_params['weight_cat_class_1'],  best_params['weight_cat_class_2']])

proba_lgbm_test = X_test[['lgbm_0', 'lgbm_1', 'lgbm_2']].to_numpy()
proba_xgb_test  = X_test[['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
# proba_cat_test  = X_test[['cat_0', 'cat_1', 'cat_2']].to_numpy()

blend_proba_test = (proba_lgbm_test * w_lgbm) + (proba_xgb_test * w_xgb) # + (proba_cat_test * w_cat)

pred = np.argmax(blend_proba_test, axis=1)

In [11]:
sub_labels = label_encoder.inverse_transform(pred)

In [12]:
np.unique(pred)

array([0, 1, 2])

# Submission

In [13]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.3.0_xgboost_lightgbm_submission.csv', index=False)

In [14]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
